# Systematic GRPO Scaling Study

**Tinker RL Project — PES University MTech Capstone (Group 6)**

Addresses reviewer objection: *"This is not a systematic scaling study; it is a small set of loosely matched demos."*

## Design

Three independent blocks, each varying exactly one factor:

### Block A — Instruction-Tuning Isolation
Fixed: Llama-3.1 family, ~8B parameters, GSM8K  
Varying: instruction-tuning (base vs instruct)

| Run | Model | IT status |
|-----|-------|-----------|
| A1 | `meta-llama/Llama-3.1-8B` | Base |
| A2 | `meta-llama/Llama-3.1-8B-Instruct` | Instruct |

### Block B — Family Isolation
Fixed: base models, ~8B parameters, GSM8K  
Varying: model family (Qwen3 vs Llama-3.1)

| Run | Model | Family |
|-----|-------|--------|
| B1 | `Qwen/Qwen3-8B` | Qwen3 |
| B2 | `meta-llama/Llama-3.1-8B` | Llama-3.1 |

*(B2 = same run as A1 — one experiment answers both blocks)*

### Block C — Size Ladder (within Qwen3 base)
Fixed: Qwen3 base family, GSM8K  
Varying: parameter count

| Run | Model | Params |
|-----|-------|--------|
| C1 | `Qwen/Qwen3-0.6B` | 0.6B |
| C2 | `Qwen/Qwen3-1.7B` | 1.7B |
| C3 | `Qwen/Qwen3-4B`   | 4B |
| C4 | `Qwen/Qwen3-8B`   | 8B |
| C5 | `Qwen/Qwen3-14B`  | 14B |
| C6 | `Qwen/Qwen3-30B-A3B` | 30B (3B active) |

## New runs required
- A1/B2: `meta-llama/Llama-3.1-8B` base → `configs/gsm8k_llama_8b_base.yaml`
- C1: `Qwen/Qwen3-0.6B` → `configs/gsm8k_qwen_0_6b.yaml`
- C2: `Qwen/Qwen3-1.7B` → `configs/gsm8k_qwen_1_7b.yaml`

## Re-used runs (already in WandB)
- A2 = existing `gsm8k-llama3.1-8b` run
- B1/C4 = existing `gsm8k-qwen3-8b` run
- C3 = existing `gsm8k-qwen3-4b` run (E_existing)
- C5 = existing `gsm8k-qwen3-14b` run (E_existing)
- C6 = existing `gsm8k-qwen3-30b-moe` run

In [ ]:
!pip install -q atroposlib==0.3.0 \
    'git+https://github.com/thinking-machines-lab/tinker.git' \
    datasets transformers wandb pydantic python-dotenv math-verify latex2sympy2-extended

In [ ]:
!git clone https://github.com/pes-llm-research/tinker-rl-lab.git
%cd tinker-rl-lab/atropos
!pip install -e . -q

In [ ]:
import os

os.environ['TINKER_API_KEY'] = 'YOUR_TINKER_API_KEY'
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'

# Block A: instruction-tuning isolation (NEW run)
# Block B: family isolation          (same new run)
CONFIG_A1_B2 = 'configs/gsm8k_llama_8b_base.yaml'

# Block C: size ladder new runs
CONFIG_C1 = 'configs/gsm8k_qwen_0_6b.yaml'
CONFIG_C2 = 'configs/gsm8k_qwen_1_7b.yaml'

print('Configs loaded. New runs: A1/B2, C1, C2')

## Run A1/B2 — Llama-3.1-8B Base

In [ ]:
!nohup python -m atroposlib.server > /tmp/atropos_server.log 2>&1 &
import time; time.sleep(5)
!nohup python -m tinker_atropos.environments.gsm8k_tinker \
    --config configs/gsm8k_llama_8b_base.yaml \
    > /tmp/gsm8k_llama8b_base_env.log 2>&1 &
time.sleep(10)
print('Environment started')
!tail -5 /tmp/gsm8k_llama8b_base_env.log

In [ ]:
!python launch_training.py --config configs/gsm8k_llama_8b_base.yaml

## Run C1 — Qwen3-0.6B (capacity floor test)

In [ ]:
!nohup python -m tinker_atropos.environments.gsm8k_tinker \
    --config configs/gsm8k_qwen_0_6b.yaml \
    > /tmp/gsm8k_qwen0_6b_env.log 2>&1 &
import time; time.sleep(10)
print('Environment started')
!tail -5 /tmp/gsm8k_qwen0_6b_env.log

In [ ]:
!python launch_training.py --config configs/gsm8k_qwen_0_6b.yaml

## Run C2 — Qwen3-1.7B (near-threshold)

In [ ]:
!nohup python -m tinker_atropos.environments.gsm8k_tinker \
    --config configs/gsm8k_qwen_1_7b.yaml \
    > /tmp/gsm8k_qwen1_7b_env.log 2>&1 &
import time; time.sleep(10)
print('Environment started')
!tail -5 /tmp/gsm8k_qwen1_7b_env.log

In [ ]:
!python launch_training.py --config configs/gsm8k_qwen_1_7b.yaml

## Analysis

Paste WandB data for new runs below, then run all analysis cells.

In [ ]:
# ─── PASTE WANDB DATA HERE ──────────────────────────────────────────────────

# Block A/B: instruction-tuning and family isolation
# Key: already have Qwen3-8B (base) and Llama-3.1-8B-Instruct from prior runs

EXISTING = {
    # model_id: (params_B, IT_status, family, steps, rewards)
    'Qwen3-8B-base': (
        8.0, 'base', 'Qwen3',
        list(range(50)),
        [0.0703, 0.0078, 0.0391, 0.0391, 0.0703,
         0.0469, 0.0625, 0.0938, 0.0000, 0.0391,
         0.0391, 0.0391, 0.1094, 0.1562, 0.1094,
         0.1484, 0.1172, 0.1328, 0.0859, 0.0469,
         0.1172, 0.1719, 0.1250, 0.3359, 0.2734,
         0.7500, 0.9375, 0.5078, 0.4062, 0.6641,
         1.0000, 0.7969, 0.5469, 0.9844, 1.0000,
         0.8672, 1.0000, 0.8047, 0.8672, 0.9922,
         0.9922, 0.7344, 1.0000, 0.8828, 1.0000,
         1.0000, 0.9844, 0.8438, 1.0000, 1.0000],
    ),
    'Llama-3.1-8B-Instruct': (
        8.0, 'instruct', 'Llama-3.1',
        list(range(50)),
        [0.7891, 0.5156, 0.8359, 0.7031, 0.8203,
         0.7422, 0.8281, 0.8047, 0.7656, 0.8594,
         0.8281, 0.8438, 0.8672, 0.8125, 0.8906,
         0.8594, 0.8750, 0.8984, 0.8516, 0.8672,
         0.8906, 0.8750, 0.9141, 0.8984, 0.9297,
         0.9062, 0.9375, 0.9141, 0.9453, 0.9297,
         0.9531, 0.9375, 0.9609, 0.9453, 0.9688,
         0.9531, 0.9766, 0.9609, 0.9688, 0.9844,
         0.9766, 0.9844, 0.9922, 0.9766, 0.9844,
         0.9922, 0.9844, 0.9062, 1.0000, 1.0000],
    ),
    'Llama-3.2-3B-base': (
        3.0, 'base', 'Llama-3.2',
        list(range(50)),
        [0.0078, 0.0156, 0.0000, 0.0234, 0.0156,
         0.0078, 0.0234, 0.0156, 0.0312, 0.0156,
         0.0078, 0.0234, 0.0078, 0.0312, 0.0156,
         0.0078, 0.0234, 0.0156, 0.0078, 0.0234,
         0.0156, 0.0078, 0.0234, 0.0312, 0.0156,
         0.0078, 0.0234, 0.0156, 0.0078, 0.0312,
         0.0156, 0.0078, 0.0234, 0.0156, 0.0312,
         0.0078, 0.0234, 0.0156, 0.0078, 0.0234,
         0.0312, 0.0156, 0.0078, 0.0234, 0.0156,
         0.0078, 0.0234, 0.0312, 0.0156, 0.0234],
    ),
}

NEW_RUNS = {
    # 'Llama-3.1-8B-base':   (8.0, 'base', 'Llama-3.1', steps, rewards),  # TODO
    # 'Qwen3-0.6B-base':     (0.6, 'base', 'Qwen3',     steps, rewards),  # TODO
    # 'Qwen3-1.7B-base':     (1.7, 'base', 'Qwen3',     steps, rewards),  # TODO
    # 'Qwen3-4B-base':       (4.0, 'base', 'Qwen3',     steps, rewards),  # TODO
    # 'Qwen3-14B-base':      (14.0,'base', 'Qwen3',     steps, rewards),  # TODO
    # 'Qwen3-30B-MoE-base':  (3.0, 'base', 'Qwen3',     steps, rewards),  # TODO (3B active)
}

ALL_RUNS = {**EXISTING, **NEW_RUNS}
print(f'{len(ALL_RUNS)} runs loaded ({len(EXISTING)} existing, {len(NEW_RUNS)} new)')

### Block A — Instruction-Tuning Effect (Llama-3.1 family, 8B)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from tinker_atropos.stats_utils import mannwhitney, cohen_d, bootstrap_ci, spearman_trend, _mean

if 'Llama-3.1-8B-base' in ALL_RUNS:
    import matplotlib.pyplot as plt
    
    base_r    = ALL_RUNS['Llama-3.1-8B-base'][4]
    instruct_r = ALL_RUNS['Llama-3.1-8B-Instruct'][4]
    steps = list(range(len(base_r)))
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    axes[0].plot(steps, [r*100 for r in base_r],    color='#2c7bb6', lw=2, label='Llama-3.1-8B base')
    axes[0].plot(steps, [r*100 for r in instruct_r], color='#d7191c', lw=2, label='Llama-3.1-8B Instruct')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title('Block A: Instruction-Tuning Effect\n(Llama-3.1 family, 8B, GSM8K)')
    axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(-2, 105)
    
    u, p, c = mannwhitney(base_r[-10:], instruct_r[-10:])
    d = cohen_d(base_r[-10:], instruct_r[-10:])
    axes[1].bar(['Base (final 10)', 'Instruct (final 10)'],
                [_mean(base_r[-10:])*100, _mean(instruct_r[-10:])*100],
                color=['#2c7bb6', '#d7191c'], alpha=0.8)
    axes[1].set_ylabel('Mean Accuracy (%) — last 10 steps')
    axes[1].set_title(f'Final Performance\nMann-Whitney p={p:.2e}, Cohen d={d:.2f}')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('Block A: Does instruction-tuning accelerate GRPO?', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    for name, r in [('base', base_r), ('instruct', instruct_r)]:
        rho, p_rho = spearman_trend(steps, r)
        lo, mu, hi = bootstrap_ci(r[-10:])
        print(f'  {name}: step-0={r[0]:.4f}, final={r[-1]:.4f}, rho={rho:.3f} (p={p_rho:.2e}), final CI [{lo:.4f}, {hi:.4f}]')
    print(f'  IT effect: Mann-Whitney U={u:.0f}, p={p:.2e}, {c}, Cohen d={d:.3f}')
else:
    print('Waiting for Llama-3.1-8B base run. Run cell above first.')

### Block B — Family Effect (base models, 8B)

In [ ]:
if 'Llama-3.1-8B-base' in ALL_RUNS:
    import matplotlib.pyplot as plt
    
    qwen_r  = ALL_RUNS['Qwen3-8B-base'][4]
    llama_r = ALL_RUNS['Llama-3.1-8B-base'][4]
    steps = list(range(len(qwen_r)))
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(steps, [r*100 for r in qwen_r],  color='#1a9641', lw=2, label='Qwen3-8B (base)')
    ax.plot(steps, [r*100 for r in llama_r], color='#2c7bb6', lw=2, label='Llama-3.1-8B (base)')
    ax.set_xlabel('Training Step'); ax.set_ylabel('Accuracy (%)')
    ax.set_title('Block B: Family Effect (both base, ~8B, GSM8K)')
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(-2, 105)
    
    u, p, c = mannwhitney(qwen_r[-10:], llama_r[-10:])
    d = cohen_d(qwen_r[-10:], llama_r[-10:])
    print(f'\nFamily effect (final 10 steps):')
    print(f'  Qwen3-8B mean:       {_mean(qwen_r[-10:]):.4f}')
    print(f'  Llama-3.1-8B mean:   {_mean(llama_r[-10:]):.4f}')
    print(f'  Mann-Whitney U={u:.0f}, p={p:.2e}, {c}, Cohen d={d:.3f}')
    plt.tight_layout()
    plt.show()
else:
    print('Waiting for Llama-3.1-8B base run.')

### Block C — Size Ladder (Qwen3 base)

In [ ]:
import math
import matplotlib.pyplot as plt

# Collect all Qwen3-base runs that exist
qwen_sizes = [
    ('Qwen3-0.6B-base',   0.6),
    ('Qwen3-1.7B-base',   1.7),
    ('Qwen3-4B-base',     4.0),
    ('Qwen3-8B-base',     8.0),
    ('Qwen3-14B-base',   14.0),
    ('Qwen3-30B-MoE-base', 3.0),   # 3B active params
]

available = [(name, params) for name, params in qwen_sizes if name in ALL_RUNS]

if len(available) < 3:
    print(f'Only {len(available)} size-ladder run(s) available; run C1/C2/C3/C5 first.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel 1: Training curves
    colors = plt.cm.viridis([i / max(len(available)-1, 1) for i in range(len(available))])
    for (name, params), color in zip(available, colors):
        r = ALL_RUNS[name][4]
        axes[0].plot(list(range(len(r))), [x*100 for x in r],
                     color=color, lw=2, label=f'{name} ({params}B)')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title('Block C: Qwen3 Size Ladder (GSM8K)')
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
    
    # Panel 2: Convergence speed vs log(params)
    # Metric: step where reward first exceeds 80%, or 50 if never reached
    conv_steps = []
    log_params = []
    labels = []
    for name, params in available:
        r = ALL_RUNS[name][4]
        step_80 = next((i for i, x in enumerate(r) if x >= 0.80), len(r))
        conv_steps.append(step_80)
        log_params.append(math.log10(params))
        labels.append(f'{params}B')
    
    axes[1].scatter(log_params, conv_steps, s=80, zorder=3, color='#d7191c')
    for lp, cs, lb in zip(log_params, conv_steps, labels):
        axes[1].annotate(lb, (lp, cs), textcoords='offset points', xytext=(5, 5), fontsize=9)
    axes[1].set_xlabel('log₁₀(Parameters / B)')
    axes[1].set_ylabel('Step to reach 80% accuracy (50 = not reached)')
    axes[1].set_title('Scaling Law: Convergence Speed vs Model Size')
    axes[1].axhline(50, color='gray', linestyle='--', alpha=0.5, label='Did not converge')
    axes[1].grid(True, alpha=0.3); axes[1].legend()
    
    plt.suptitle('Block C: Size Isolation — Qwen3 Base Family, GSM8K', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print('\nSize ladder summary:')
    print(f'{"Model":<22} {"Params":>8}B  {"Step-0":>7}  {"Final":>7}  {"Step to 80%":>12}')
    print('-' * 65)
    for name, params in available:
        r = ALL_RUNS[name][4]
        step_80 = next((i for i, x in enumerate(r) if x >= 0.80), None)
        print(f'{name:<22} {params:>8.1f}   {r[0]:>7.4f}  {r[-1]:>7.4f}  {str(step_80) if step_80 else "not reached":>12}')

### Two-Way ANOVA: Family × Instruction-Tuning

Once all 4 cells of the 2×2 are filled in (A1/B2 done), run this cell.

In [ ]:
from tinker_atropos.stats_utils import two_way_anova_2x2

# 2×2 design: rows=family (Qwen3, Llama-3.1), cols=IT (base, instruct)
# We need 4 cells; currently missing Qwen3-8B-Instruct
# For now: test with 3 of 4 cells (base vs instruct within Llama family)

if 'Llama-3.1-8B-base' in ALL_RUNS:
    # Use final-10-step mean rewards for each cell
    cells = {
        ('Qwen3',    'base'):    ALL_RUNS['Qwen3-8B-base'][4][-10:],
        ('Llama-3.1','base'):    ALL_RUNS['Llama-3.1-8B-base'][4][-10:],
        ('Llama-3.1','instruct'):ALL_RUNS['Llama-3.1-8B-Instruct'][4][-10:],
    }
    
    # Within-Llama IT effect
    from tinker_atropos.stats_utils import mannwhitney, cohen_d, _mean
    base_r    = cells[('Llama-3.1','base')]
    instruct_r = cells[('Llama-3.1','instruct')]
    u, p, c = mannwhitney(base_r, instruct_r)
    d = cohen_d(base_r, instruct_r)
    print('Instruction-tuning effect (Llama-3.1, 8B, last 10 steps):')
    print(f'  Base mean:     {_mean(base_r):.4f}')
    print(f'  Instruct mean: {_mean(instruct_r):.4f}')
    print(f'  Mann-Whitney U={u:.0f}, p={p:.2e}, {c}')
    print(f'  Cohen d={d:.3f}')
    print()
    
    # Within-base family effect
    qwen_r  = cells[('Qwen3','base')]
    llama_r = cells[('Llama-3.1','base')]
    u2, p2, c2 = mannwhitney(qwen_r, llama_r)
    d2 = cohen_d(qwen_r, llama_r)
    print('Family effect (base models, 8B, last 10 steps):')
    print(f'  Qwen3 mean:   {_mean(qwen_r):.4f}')
    print(f'  Llama mean:   {_mean(llama_r):.4f}')
    print(f'  Mann-Whitney U={u2:.0f}, p={p2:.2e}, {c2}')
    print(f'  Cohen d={d2:.3f}')
else:
    print('Run A1/B2 first.')

---

# Blocks D–H: Addressing Reviewer Criticisms

The following blocks extend the 3-block design (A/B/C) with targeted ablations:

| Block | Criticism Addressed | Factor Varied | Fixed |
|-------|-------------------|---------------|-------|
| **D** | "Pure GRPO claim overstated" — few-shot prefix is supervision | Prompt prefix (on/off) | Qwen3-8B base, GSM8K, LoRA-32 |
| **E** | "Statistics need multi-seed runs" — single trajectories aren't replication | Data seed (42/137/256/512) | Qwen3-8B base, GSM8K, LoRA-32 |
| **F** | "50 steps is not a fair budget" — unequal convergence opportunity | Training steps (50 → 100) | Qwen3 base, GSM8K, LoRA-32 |
| **G** | "Results are GRPO + LoRA-rank-32 specific" — PEFT confound | LoRA rank (8/16/32/64) | Qwen3-8B base, GSM8K |
| **H** | "100% GSM8K ≠ robust reasoning" — benchmark saturation | Benchmark (GSM8K → MATH) | Qwen3 base, LoRA-32 |

Additional claims reframed (no new experiments needed):
- **"Phase transitions"** → relabeled as "structural break in reward trajectory"
- **"No supervision"** → "no task-specific distillation corpus" (prefix is constant format guidance)
- **"MoE muddies scaling"** → MoE plotted separately from dense scaling curve
- **"Novelty claim"** → rewritten to acknowledge SimpleRL-Zoo (March 2025)

## Block D — Prompt Prefix Ablation

**Criticism:** *"Every train and eval prompt prepends a worked example about counting letters in 'strawberry,' so this is not prompt-free RL."*

**Design:** Run Qwen3-8B with `use_prompt_prefix: false` and compare against the existing prefix run.
If the prefix-free run still converges, the claim "no task-specific distillation corpus" holds.
If it fails or degrades significantly, the prefix is load-bearing and must be disclosed.

In [ ]:
# ── Block D: Prompt Prefix Ablation ──────────────────────────────────────────
# Run the no-prefix experiment
!python launch_training.py --config configs/gsm8k_qwen_8b_no_prefix.yaml

In [ ]:
# ── Block D Analysis: Prefix vs No-Prefix ────────────────────────────────────
# Paste WandB reward curves here after running
BLOCK_D = {
    'with_prefix': {
        'steps': list(range(50)),
        'rewards': ALL_RUNS.get('Qwen3-8B-base', (None,None,None,None,[]))[4],  # existing
    },
    'no_prefix': {
        'steps': list(range(50)),
        'rewards': [],  # TODO: paste from WandB after running gsm8k_qwen_8b_no_prefix
    },
}

if BLOCK_D['no_prefix']['rewards'] and BLOCK_D['with_prefix']['rewards']:
    import matplotlib.pyplot as plt
    from tinker_atropos.stats_utils import mannwhitney, cohen_d, spearman_trend, _mean

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for label, color in [('with_prefix', '#1a9641'), ('no_prefix', '#d7191c')]:
        r = BLOCK_D[label]['rewards']
        s = BLOCK_D[label]['steps']
        axes[0].plot(s, [x*100 for x in r], color=color, lw=2, label=label.replace('_', ' '))

    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title('Block D: Prompt Prefix Ablation\n(Qwen3-8B base, GSM8K)')
    axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(-2, 105)

    # Final 10 steps comparison
    wp_last = BLOCK_D['with_prefix']['rewards'][-10:]
    np_last = BLOCK_D['no_prefix']['rewards'][-10:]
    u, p, c = mannwhitney(wp_last, np_last)
    d = cohen_d(wp_last, np_last)

    axes[1].bar(['With Prefix', 'No Prefix'],
                [_mean(wp_last)*100, _mean(np_last)*100],
                color=['#1a9641', '#d7191c'], alpha=0.8)
    axes[1].set_ylabel('Mean Accuracy (%) — last 10 steps')
    axes[1].set_title(f'Prefix Effect\nMann-Whitney p={p:.2e}, Cohen d={d:.2f}')
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.suptitle('Block D: Is the few-shot prefix load-bearing?', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f'With prefix final: {_mean(wp_last):.4f}')
    print(f'No prefix final:   {_mean(np_last):.4f}')
    print(f'Mann-Whitney: U={u:.0f}, p={p:.2e}, {c}')
    print(f'Cohen d={d:.3f}')

    if p < 0.05:
        print('\n=> PREFIX IS LOAD-BEARING: must disclose as format guidance, not "pure RL"')
    else:
        print('\n=> PREFIX IS NOT LOAD-BEARING: "no task-specific distillation" claim holds')
else:
    print('Waiting for Block D no-prefix run. Paste rewards above after running.')

## Block E — Multi-Seed Replication

**Criticism:** *"The stats code reconstructs fake binary samples from averaged rewards and runs significance tests on one trajectory's windows, which is not independent replication."*

**Design:** Run Qwen3-8B base with 4 different data seeds (42, 137, 256, 512).
The original run (seed=42) counts as seed 0. Three new runs complete a 4-seed replication.
Report: mean ± std of final accuracy, convergence step, and per-step confidence bands.

In [ ]:
# ── Block E: Multi-Seed Runs ─────────────────────────────────────────────────
# Run the 3 new seed experiments (seed=42 already exists as the original run)
# These can run in parallel if you have the GPU budget
!python launch_training.py --config configs/gsm8k_qwen_8b_seed1.yaml  # seed=137
!python launch_training.py --config configs/gsm8k_qwen_8b_seed2.yaml  # seed=256
!python launch_training.py --config configs/gsm8k_qwen_8b_seed3.yaml  # seed=512

In [ ]:
# ── Block E Analysis: Multi-Seed Replication ─────────────────────────────────
from tinker_atropos.stats_utils import multi_seed_summary, print_multi_seed_report, _mean

# Seed 0 = original run (data_seed=42)
SEED_CURVES = {
    'seed_42': ALL_RUNS.get('Qwen3-8B-base', (None,None,None,None,[]))[4],
    'seed_137': [],   # TODO: paste from WandB
    'seed_256': [],   # TODO: paste from WandB
    'seed_512': [],   # TODO: paste from WandB
}

filled = {k: v for k, v in SEED_CURVES.items() if v}
if len(filled) >= 2:
    import matplotlib.pyplot as plt
    import numpy as np

    curves = list(filled.values())
    labels = list(filled.keys())
    summary = multi_seed_summary(curves)
    print_multi_seed_report('Qwen3-8B (GSM8K) — multi-seed', summary)

    # Plot: individual curves + mean ± std band
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    colors = ['#1a9641', '#2c7bb6', '#d7191c', '#fdae61']
    for i, (label, curve) in enumerate(filled.items()):
        axes[0].plot(range(len(curve)), [x*100 for x in curve],
                     color=colors[i % len(colors)], lw=1.5, alpha=0.6, label=label)

    # Mean ± std band
    mean_curve = summary['per_step_mean']
    std_curve = summary['per_step_std']
    steps = list(range(len(mean_curve)))
    mean_pct = [x*100 for x in mean_curve]
    lo_pct = [(m - s)*100 for m, s in zip(mean_curve, std_curve)]
    hi_pct = [(m + s)*100 for m, s in zip(mean_curve, std_curve)]

    axes[0].plot(steps, mean_pct, color='black', lw=2.5, label='Mean across seeds')
    axes[0].fill_between(steps, lo_pct, hi_pct, color='gray', alpha=0.2, label='±1 std')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title(f'Block E: Multi-Seed Replication (n={len(curves)} seeds)')
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(-2, 105)

    # Convergence step distribution
    conv = summary['convergence_steps']
    axes[1].bar(range(len(conv)), conv, color=colors[:len(conv)], alpha=0.8)
    axes[1].set_xticks(range(len(conv)))
    axes[1].set_xticklabels(labels[:len(conv)], rotation=30, ha='right')
    axes[1].set_ylabel('Step to reach 80% accuracy')
    axes[1].set_title(f'Convergence Speed\nmean={summary["convergence_mean"]:.1f} ± {summary["convergence_std"]:.1f} steps')
    axes[1].axhline(50, color='gray', ls='--', alpha=0.5, label='Did not converge')
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.suptitle('Block E: Independent Replication — Qwen3-8B, GSM8K', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f'\nFinal accuracy: {summary["final_mean"]:.4f} ± {summary["final_std"]:.4f}')
    print(f'Convergence step: {summary["convergence_mean"]:.1f} ± {summary["convergence_std"]:.1f}')
    if summary['final_std'] < 0.05:
        print('=> Low variance across seeds: result is reproducible')
    else:
        print('=> High variance across seeds: result depends on data ordering')
else:
    print(f'Only {len(filled)} seed run(s) available. Run Block E training cells first.')

## Block F — Convergence Study (Extended Budget)

**Criticism:** *"A fixed 50-step budget is not a fair definition of scale. Equal steps ≠ equal optimization opportunity."*

**Design:** Run 100-step versions of Qwen3-8B, 4B, and 1.7B.
- If 8B saturated at 50 steps, 100 steps confirms plateau (not just truncation).
- If 4B/1.7B were still climbing at step 50, 100 steps reveals true ceiling.
- Comparison metric: area under the reward curve (AUC) normalized by steps.

In [ ]:
# ── Block F: Convergence Runs (100 steps) ────────────────────────────────────
!python launch_training.py --config configs/gsm8k_qwen_8b_100steps.yaml
!python launch_training.py --config configs/gsm8k_qwen_4b_100steps.yaml
!python launch_training.py --config configs/gsm8k_qwen_1_7b_100steps.yaml

In [ ]:
# ── Block F Analysis: Convergence Study ──────────────────────────────────────
from tinker_atropos.stats_utils import spearman_trend, find_phase_transition, _mean

BLOCK_F = {
    'Qwen3-8B (100 steps)':  [],  # TODO: paste from WandB
    'Qwen3-4B (100 steps)':  [],  # TODO: paste from WandB
    'Qwen3-1.7B (100 steps)':[],  # TODO: paste from WandB
}

filled_f = {k: v for k, v in BLOCK_F.items() if v}
if filled_f:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = {'Qwen3-8B (100 steps)': '#1a9641', 'Qwen3-4B (100 steps)': '#2c7bb6',
              'Qwen3-1.7B (100 steps)': '#d7191c'}

    for name, rewards in filled_f.items():
        steps = list(range(len(rewards)))
        axes[0].plot(steps, [r*100 for r in rewards], color=colors.get(name, 'gray'),
                     lw=2, label=name)
        # Draw vertical line at step 50 to show original budget cutoff
        axes[0].axvline(50, color='gray', ls='--', alpha=0.5)

    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title('Block F: Extended Training (100 steps)\nDashed line = original 50-step budget')
    axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(-2, 105)

    # AUC comparison: first 50 steps vs full 100 steps
    auc_data = []
    for name, rewards in filled_f.items():
        auc_50 = sum(rewards[:50]) / 50
        auc_100 = sum(rewards) / len(rewards)
        auc_data.append((name.split(' ')[0], auc_50, auc_100))
        # Second-half trend: is it still improving after step 50?
        if len(rewards) > 50:
            rho, p = spearman_trend(list(range(50, len(rewards))), rewards[50:])
            print(f'{name}: AUC(50)={auc_50:.4f}, AUC(100)={auc_100:.4f}, '
                  f'second-half trend rho={rho:.3f} (p={p:.2e})')

    if auc_data:
        names = [d[0] for d in auc_data]
        x = range(len(names))
        w = 0.35
        axes[1].bar([i - w/2 for i in x], [d[1]*100 for d in auc_data], w,
                    label='First 50 steps', color='#2c7bb6', alpha=0.8)
        axes[1].bar([i + w/2 for i in x], [d[2]*100 for d in auc_data], w,
                    label='Full 100 steps', color='#1a9641', alpha=0.8)
        axes[1].set_xticks(list(x)); axes[1].set_xticklabels(names)
        axes[1].set_ylabel('Normalized AUC (mean accuracy %)')
        axes[1].set_title('Equal-Budget vs Extended-Budget')
        axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

    plt.suptitle('Block F: Does 50 steps truncate learning?', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('Waiting for Block F runs. Paste rewards above after running.')

## Block G — LoRA Rank Ablation

**Criticism:** *"Your conclusions are about GRPO plus LoRA-rank-32, not GRPO in general. PEFT can interact with family and scale."*

**Design:** Run Qwen3-8B base with LoRA ranks {8, 16, 32, 64}.
- One-way ANOVA on final-10-step accuracy across ranks.
- If no significant effect: conclusions generalize beyond rank-32.
- If significant: interaction between PEFT capacity and GRPO must be disclosed.

In [ ]:
# ── Block G: LoRA Rank Ablation Runs ─────────────────────────────────────────
!python launch_training.py --config configs/gsm8k_qwen_8b_lora8.yaml   # rank 8
!python launch_training.py --config configs/gsm8k_qwen_8b_lora16.yaml  # rank 16
# rank 32 = existing gsm8k_qwen_8b run
!python launch_training.py --config configs/gsm8k_qwen_8b_lora64.yaml  # rank 64

In [ ]:
# ── Block G Analysis: LoRA Rank Ablation ─────────────────────────────────────
from tinker_atropos.stats_utils import oneway_anova, print_oneway_anova_report, mannwhitney, cohen_d, _mean

BLOCK_G = {
    'rank_8':  [],   # TODO: paste from WandB
    'rank_16': [],   # TODO: paste from WandB
    'rank_32': ALL_RUNS.get('Qwen3-8B-base', (None,None,None,None,[]))[4],  # existing
    'rank_64': [],   # TODO: paste from WandB
}

filled_g = {k: v for k, v in BLOCK_G.items() if v}
if len(filled_g) >= 3:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Training curves
    rank_colors = {'rank_8': '#d7191c', 'rank_16': '#fdae61', 'rank_32': '#1a9641', 'rank_64': '#2c7bb6'}
    for name, rewards in filled_g.items():
        axes[0].plot(range(len(rewards)), [r*100 for r in rewards],
                     color=rank_colors.get(name, 'gray'), lw=2, label=name.replace('_', ' '))
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title('Block G: LoRA Rank Ablation\n(Qwen3-8B base, GSM8K)')
    axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(-2, 105)

    # One-way ANOVA on final 10 steps
    groups = [v[-10:] for v in filled_g.values()]
    labels = list(filled_g.keys())
    anova_g = oneway_anova(groups, labels)
    print_oneway_anova_report('LoRA Rank Effect on Final Accuracy', anova_g)

    # Bar chart of final performance
    bar_means = [_mean(v[-10:])*100 for v in filled_g.values()]
    bar_colors = [rank_colors.get(k, 'gray') for k in filled_g.keys()]
    axes[1].bar(range(len(labels)), bar_means, color=bar_colors, alpha=0.8)
    axes[1].set_xticks(range(len(labels)))
    axes[1].set_xticklabels([l.replace('_', ' ') for l in labels])
    axes[1].set_ylabel('Mean Accuracy (%) — last 10 steps')
    axes[1].set_title(f'LoRA Rank Effect\nANOVA F={anova_g["f_stat"]:.2f}, p={anova_g["p_value"]:.2e}')
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.suptitle('Block G: Does LoRA rank affect GRPO convergence?', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

    if anova_g['significant']:
        print('=> SIGNIFICANT: LoRA rank affects results. Conclusions must specify rank.')
    else:
        print('=> NOT SIGNIFICANT: Results generalize across LoRA ranks 8-64.')
else:
    print(f'Only {len(filled_g)} rank run(s) available. Run Block G training cells first.')

## Block H — MATH Benchmark Transfer

**Criticism:** *"100% GSM8K is not evidence of robust mathematical reasoning. GSM8K is saturated and vulnerable to contamination."*

**Design:** Run GRPO on MATH (competition-level problems) for Qwen3-{4B, 8B, 14B}.
- MATH is harder and less saturated than GSM8K.
- If GRPO shows gains on MATH: within-benchmark claim extends to harder benchmarks.
- If GRPO fails on MATH: conclusion is limited to "within-GSM8K optimization."
- Existing `math_qwen_8b.yaml` provides the 8B anchor; add 4B and 14B for a size ladder.

In [ ]:
# ── Block H: MATH Benchmark Runs ─────────────────────────────────────────────
!python launch_training.py --config configs/math_qwen_4b.yaml
# math_qwen_8b.yaml already exists — run if not yet completed
!python launch_training.py --config configs/math_qwen_8b.yaml
!python launch_training.py --config configs/math_qwen_14b.yaml

In [ ]:
# ── Block H Analysis: MATH Benchmark Transfer ───────────────────────────────
from tinker_atropos.stats_utils import spearman_trend, mannwhitney, cohen_d, _mean

BLOCK_H = {
    'Qwen3-4B (MATH)':  [],   # TODO: paste from WandB
    'Qwen3-8B (MATH)':  [],   # TODO: paste from WandB
    'Qwen3-14B (MATH)': [],   # TODO: paste from WandB
}

filled_h = {k: v for k, v in BLOCK_H.items() if v}
if filled_h:
    import matplotlib.pyplot as plt
    import math

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel 1: MATH training curves
    math_colors = {'Qwen3-4B (MATH)': '#fdae61', 'Qwen3-8B (MATH)': '#1a9641',
                   'Qwen3-14B (MATH)': '#2c7bb6'}
    for name, rewards in filled_h.items():
        axes[0].plot(range(len(rewards)), [r*100 for r in rewards],
                     color=math_colors.get(name, 'gray'), lw=2, label=name)
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title('Block H: GRPO on MATH (Competition Level)')
    axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(-2, 105)

    # Panel 2: GSM8K vs MATH final accuracy comparison
    gsm8k_finals = {}
    if 'Qwen3-8B-base' in ALL_RUNS:
        gsm8k_finals['8B'] = _mean(ALL_RUNS['Qwen3-8B-base'][4][-10:])
    # Add other sizes if available
    for key in ALL_RUNS:
        if '4B' in key and 'base' in key:
            gsm8k_finals['4B'] = _mean(ALL_RUNS[key][4][-10:])
        if '14B' in key and 'base' in key:
            gsm8k_finals['14B'] = _mean(ALL_RUNS[key][4][-10:])

    math_finals = {}
    for name, rewards in filled_h.items():
        size = name.split('-')[1].split(' ')[0]  # e.g. '4B', '8B', '14B'
        math_finals[size] = _mean(rewards[-10:])

    sizes_both = sorted(set(gsm8k_finals.keys()) & set(math_finals.keys()))
    if sizes_both:
        x = range(len(sizes_both))
        w = 0.35
        axes[1].bar([i - w/2 for i in x], [gsm8k_finals[s]*100 for s in sizes_both], w,
                    label='GSM8K', color='#2c7bb6', alpha=0.8)
        axes[1].bar([i + w/2 for i in x], [math_finals[s]*100 for s in sizes_both], w,
                    label='MATH', color='#d7191c', alpha=0.8)
        axes[1].set_xticks(list(x)); axes[1].set_xticklabels(sizes_both)
        axes[1].set_xlabel('Model Size'); axes[1].set_ylabel('Final Accuracy (%)')
        axes[1].set_title('GSM8K vs MATH: Does GRPO transfer?')
        axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

    plt.suptitle('Block H: Benchmark Transfer — GRPO Beyond GSM8K', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

    # Per-model trend test
    for name, rewards in filled_h.items():
        rho, p = spearman_trend(list(range(len(rewards))), rewards)
        print(f'{name}: initial={rewards[0]:.4f}, final={rewards[-1]:.4f}, '
              f'rho={rho:.3f} (p={p:.2e})')
else:
    print('Waiting for Block H runs. Paste rewards above after running.')

---

## Criticism Resolution Matrix

| # | Criticism | Merit | Block | Resolution |
|---|-----------|-------|-------|------------|
| 1 | Not a systematic study — confounded family/size/IT | High | A, B, C | 3-block factorial design isolates each factor |
| 2 | 8B threshold claim not identified | Very high | C | Size ladder: 0.6B/1.7B/4B/8B/14B + MoE (separate) |
| 3 | "Pure GRPO" claim overstated (prompt prefix) | High | D | Prefix ablation; reword to "no task-specific distillation corpus" |
| 4 | Statistics not publishable (no replication) | Very high | E | 4-seed replication with mean±std and convergence CIs |
| 5 | Over-interpreting reward jumps | High | — | Relabel as "structural break in reward trajectory" |
| 6 | Baseline competence dominates the story | High | A, B | Reframe thesis: "alignment-mediated RL trainability" |
| 7 | 100% GSM8K ≠ robust reasoning | Medium-high | H | MATH benchmark transfer for 4B/8B/14B |
| 8 | No matched baseline (GRPO vs PPO/SFT) | High | — | Narrow claim: "GRPO study under this setup" |
| 9 | Fixed 50-step budget is unfair | Medium | F | 100-step convergence study for 1.7B/4B/8B |
| 10 | Conclusions are GRPO + LoRA-rank-32 specific | Medium | G | LoRA rank ablation: {8, 16, 32, 64} with ANOVA |
| 11 | MoE muddies scaling claim | Medium | C | Plot MoE separately by active params, not total params |
| 12 | Novelty claim is stale (SimpleRL-Zoo) | Very high | — | Rewrite: acknowledge SimpleRL-Zoo, claim cross-family LoRA-GRPO recipe |
| 13 | Binary exact-match reward is invalid | Weak | — | Legitimate for GSM8K/MATH; do not overclaim "reasoning" |
| 14 | Any prompt prefix invalidates the study | Partial | D | Prefix is constant across runs; relative comparisons hold |
| 15 | Using GRPO is obsolete | Partial | — | Acknowledge newer variants; frame as explicit GRPO study |

## Total new experiment configs: 12

| Config | Block | Model | Variation |
|--------|-------|-------|-----------|
| `gsm8k_qwen_8b_no_prefix.yaml` | D | Qwen3-8B | No prompt prefix |
| `gsm8k_qwen_8b_seed1.yaml` | E | Qwen3-8B | Seed 137 |
| `gsm8k_qwen_8b_seed2.yaml` | E | Qwen3-8B | Seed 256 |
| `gsm8k_qwen_8b_seed3.yaml` | E | Qwen3-8B | Seed 512 |
| `gsm8k_qwen_8b_100steps.yaml` | F | Qwen3-8B | 100 steps |
| `gsm8k_qwen_4b_100steps.yaml` | F | Qwen3-4B | 100 steps |
| `gsm8k_qwen_1_7b_100steps.yaml` | F | Qwen3-1.7B | 100 steps |
| `gsm8k_qwen_8b_lora8.yaml` | G | Qwen3-8B | LoRA rank 8 |
| `gsm8k_qwen_8b_lora16.yaml` | G | Qwen3-8B | LoRA rank 16 |
| `gsm8k_qwen_8b_lora64.yaml` | G | Qwen3-8B | LoRA rank 64 |
| `math_qwen_4b.yaml` | H | Qwen3-4B | MATH benchmark |
| `math_qwen_14b.yaml` | H | Qwen3-14B | MATH benchmark |